# KU-SBB break detection: reproduce the submission

This notebook rebuilds `submission.csv` from the raw competition test series using the ten trained models saved in
`models/`. **It does not retrain anything.**

Each series is a run of numbers split at a known point into a *before* stretch and an *after* stretch. The models
estimate the probability that the process genuinely changed after that point. Run the cells from top to bottom; no
manual steps are needed.

Expected runtime on 12 CPU cores is about 20–25 minutes, almost all of it in step 3. The last cell checks that the
written file is byte-for-byte the submitted one (md5 `dc335464fec173ae2cf42996dfdfb67b`).

## 1. Environment and version check
We first make the project's code importable and compare the installed package versions with the ones the submission
was produced with.

A different version still runs and ranks the series the same way. It can, however, change the last printed digits of
some probabilities, and then the md5 check at the end would not match. That case is reported as a warning, not an
error.

In [1]:
import os, sys, time, hashlib
from pathlib import Path
from importlib.metadata import version
ROOT = Path.cwd()                                   # the notebook lives in the SUBMISSION folder
assert (ROOT / "src" / "paths.py").exists(), "open this notebook from inside the SUBMISSION folder"
for v in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS"):
    os.environ.setdefault(v, "1")                   # one numeric thread per worker process (avoids CPU oversubscription)
sys.path.insert(0, str(ROOT / "src"))
PINNED = {"numpy": "2.3.5", "pandas": "2.3.3", "scipy": "1.16.3", "scikit-learn": "1.8.0",
          "lightgbm": "4.7.0", "joblib": "1.5.3", "pyarrow": "26.0.0"}
print("Python", sys.version.split()[0], "(verified on 3.13.14)")
for pkg, want in PINNED.items():
    have = version(pkg)
    print(f"  {pkg:13s} {have:10s}" + ("" if have == want else f"  WARNING: pinned {want}; the last digits may differ"))

Python 3.13.14 (verified on 3.13.14)
  numpy         2.3.5     
  pandas        2.3.3     
  scipy         1.16.3    
  scikit-learn  1.8.0     
  lightgbm      4.7.0     
  joblib        1.5.3     
  pyarrow       26.0.0    


## 2. Where is the competition data?
The data is not shipped with the submission. Its location is the single setting `data_dir` in
`config/settings.json`. The default, `../data`, is a folder named `data` next to this one. If a file is missing,
this cell stops with a message naming it.

In [2]:
import numpy as np, pandas as pd
import paths                                        # reads config/settings.json
need = ["test_series.parquet", "sample_submission.csv"]
missing = [f for f in need if not (paths.DATA / f).exists()]
if missing:
    raise FileNotFoundError(f"missing {missing} in the configured data folder; set data_dir in config/settings.json")
print("data folder found; files present:", need)

data folder found; files present: ['test_series.parquet', 'sample_submission.csv']


## 3. Feature extraction (the slow step)
Each series becomes 781 numbers ("features"), computed from that series alone; no other series is ever looked at.

- **Scaling.** Every series is first rescaled with the median and spread of its *before* stretch, so features do not
  depend on units or offsets.
- **Both directions.** The features are computed twice: once reading the series forwards, and once reading it
  backwards, with the boundary mirrored.
- **Caching.** Results go into `cache/` so a rerun is fast. Delete `cache/` to start fresh. It holds features of the
  test data, so it is never distributed.

The ten feature families are described in `REPORT.md`, section 1.

In [3]:
import json, extract, views
FS = json.load(open(paths.CONFIG / "featset_v6.json"))   # the families and the exact 781 columns, in model order
ids, series = extract.load_series(paths.DATA / "test_series.parquet")
print(f"{len(series)} test series loaded")
t0 = time.time(); orig, rev = [], []
for i, fam in enumerate(FS["fams"], 1):
    orig.append(extract.extract_family(paths.DATA / "test_series.parquet", fam, series, ids))   # forwards view
    rev.append(views.fam_view("test_series", fam, "rev", series, ids))                         # backwards view
    print(f"  [{i:2d}/{len(FS['fams'])}] {fam:8s} done   elapsed {time.time() - t0:6.0f} s", flush=True)
drop = ["shape__cycres_mean_shift"]                 # a column removed from every model since v1 (numerically zero)
Xo = pd.concat(orig, axis=1).drop(columns=drop, errors="ignore")
Xr = pd.concat(rev, axis=1).drop(columns=drop, errors="ignore")
print("feature tables:", Xo.shape, Xr.shape)

4000 test series loaded


[extract] test_series wide: (4000, 181) in 24.4s


[view] test_series wide rev: (4000, 181) nan->0: 0 13s


  [ 1/10] wide     done   elapsed     38 s


[extract] test_series shape: (4000, 77) in 5.1s


[view] test_series shape rev: (4000, 77) nan->0: 0 4s


... (22 outputs left out)

[extract] test_series dgplr2: (4000, 10) in 261.1s


[view] test_series dgplr2 rev: (4000, 10) nan->0: 0 247s


  [10/10] dgplr2   done   elapsed   2144 s


feature tables: (4000, 872) (4000, 872)


## 4. Load the ten saved models (the checkpoint)
The model is LightGBM gradient-boosted trees: many small decision trees, each correcting the previous ones. It was
trained ten times with seeds 0–9, and the final prediction averages all ten.

The files are `models/lgb_t56_0.txt` … `models/lgb_t56_9.txt`. `models/recipe2.json` records their settings and
the column order. Loading them is all the prediction needs.

In [4]:
from recipe2 import Recipe2
R = Recipe2.load(paths.MODELS)
print("boosters loaded:", sum(len(b) for b in R.boosters.values()), "| columns expected:", len(R.cols))

boosters loaded: 10 | columns expected: 781


## 5. Predict, averaging the forwards and backwards readings
For every series, each of the ten models scores the forwards reading and the backwards reading. The two scores are
averaged, and then the ten models are averaged.

Training used both readings as well. A lasting change is still a lasting change when the series is read backwards,
so the backwards copy is another valid example with the same answer. This symmetry was the single largest
improvement in the project.

In [5]:
tids = pd.read_csv(paths.DATA / "sample_submission.csv", dtype={"id": str}).id.values   # official id order
To, Tr = Xo.loc[tids][R.cols], Xr.loc[tids][R.cols]
p = R.predict(To, Tr)                               # mean over 10 models of 0.5*(forwards + backwards), clipped to [0,1]
print("predicted", len(p), "series; mean probability", round(float(p.mean()), 4))

predicted 4000 series; mean probability 0.4699


## 6. The flat-series rule
A series whose values barely move (range ≤ 1e-9 × max(1, |median|)) has nothing to detect, so it gets exactly 0.5.
There are none in the test set, but the rule is applied for completeness.

In [6]:
from recipe2 import is_near_constant
pos = {k: j for j, k in enumerate(ids)}
flat = np.array([is_near_constant(series[pos[i]][0]) for i in tids])
p = np.where(flat, 0.5, p)
print("flat series in the test set:", int(flat.sum()))

flat series in the test set: 0


## 7. Write `submission.csv`
This cell writes the submission file. It has two columns, `id` and `break_probability`, one row per test series, in
the official order. It is written with Unix line endings and full precision, so it reproduces byte for byte.

In [7]:
from subcheck import write_checked
OUTPUT = ROOT / "submission.csv"
write_checked(pd.Series(p, index=tids), OUTPUT)     # also re-reads the file and checks ids, order and range

[submission] submission.csv: 4000 rows OK, mean=0.4699


## 8. Final validation
We check the row count, the id set and order, the value range and the md5 of the file against the submitted one.

In [8]:
s = pd.read_csv(OUTPUT, dtype={"id": str}, float_precision="round_trip")
ss = pd.read_csv(paths.DATA / "sample_submission.csv", dtype={"id": str})
v = s.break_probability.to_numpy()
checks = {"rows == 4000": len(s) == 4000, "columns": list(s.columns) == ["id", "break_probability"],
          "ids once each, official order": bool(s.id.is_unique and (s.id.values == ss.id.values).all()),
          "finite and within [0,1]": bool(np.isfinite(v).all() and (v >= 0).all() and (v <= 1).all())}
md5 = hashlib.md5(OUTPUT.read_bytes()).hexdigest()
checks["md5 == dc335464fec173ae2cf42996dfdfb67b"] = md5 == "dc335464fec173ae2cf42996dfdfb67b"
for k, ok in checks.items():
    print(("PASS " if ok else "FAIL ") + k)
print("md5", md5)
assert all(checks.values()), 'see REPORT.md, section 4 (version lock), if only the md5 differs'
print('all checks passed')

PASS rows == 4000
PASS columns
PASS ids once each, official order
PASS finite and within [0,1]
PASS md5 == dc335464fec173ae2cf42996dfdfb67b
md5 dc335464fec173ae2cf42996dfdfb67b
all checks passed


## Optional: retrain from scratch (disabled)
Retraining is **not** needed to reproduce the submission. It recomputes the features of all 24,000 training series
and refits the ten models, then compares the new predictions with `submission.csv`. Expect about 3–4 hours on 12
idle cores.

The command writes only into the work folder you name. It never overwrites `models/` or `submission.csv`. Set
`RETRAIN = True` to run it.

In [9]:
RETRAIN = False
if RETRAIN:
    import subprocess
    subprocess.run([sys.executable, "run_all.py", "--out", str(ROOT.parent / "kusbb_rebuild_work")], check=True)
else:
    print("retraining skipped (RETRAIN = False)")

retraining skipped (RETRAIN = False)
